In [1]:
from vizdoom import *
import cv2
import numpy as np
import time
import random
from matplotlib import pyplot as plt
import pandas as pd
import os
from tqdm import tqdm

# Gym imports
import gymnasium as gym
from gymnasium import Env
from gymnasium.spaces import Discrete, Box

# Ollama imports
import requests
import json

# Hashing import fot the llm caching
import hashlib
from functools import lru_cache

# Stable Baseline loading
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from stable_baselines3.common.monitor import Monitor

In [2]:
# Instantiate TakeCover environment
CONFIG_PATH = './github/ViZDoom/scenarios/take_cover.cfg'
CACHE_PATH = './logs/policy_cache.json'

In [3]:
# Create Vizdoom OpenAI Gym Environment
class VizDoomGym(Env): 
    def __init__(self, render=False): 
        # Setup the game 
        super().__init__()
        self.frame_skip = 4
        self.game = DoomGame()
        self.game.load_config(CONFIG_PATH)
        
        # Render frame logic
        self.game.set_window_visible(render)
        
        # Start the game 
        self.game.init()
        
        # Create the action space and observation space
        self.observation_space = Box(low=0, high=255, shape=(100,160,1), dtype=np.uint8) 
        self.action_space = Discrete(2) # Move left, move right
        self._actions = np.eye(2, dtype=np.uint8)

        # Basic Reward Shaping
        self.game.set_living_reward(1)
        self.game.set_hit_taken_penalty(1)
        self.game.set_death_penalty(5)
        
    # This is how we take a step in the environment
    def step(self, action):
        # Specify action and take step 
        reward = self.game.make_action(self._actions[action].tolist(), self.frame_skip) 
        
        # Get the new state of the game and check if it's done
        if self.game.get_state(): 
            obs    = self._process_frame(self.game.get_state().screen_buffer)
            health = self.game.get_state().game_variables[0]
            info   = {"health": health}
        else: 
            obs = np.zeros(self.observation_space.shape, dtype=np.uint8)
            info = {"health": 0}
        
        terminated = self.game.is_episode_finished()
        if terminated:
            info["survival_time_tics"] = self.game.get_episode_time()
            
        truncated = False

        return obs, reward, terminated, truncated, info
    
    # Define how to render the game or environment 
    def render(self): 
        pass
    
    # Starting a new game 
    def reset(self, seed=None, options=None): 
        super().reset(seed=seed)
        self.game.new_episode()
        state = self.game.get_state()

        if state is not None:
            obs = self._process_frame(state.screen_buffer)
        else:
            obs = np.zeros(self.observation_space.shape, dtype=np.uint8)

        return obs, {}
    
    # Call to close down the game
    def close(self): 
        self.game.close()

    def _process_frame(self, buffer: np.ndarray):
        hwc  = np.moveaxis(buffer, 0, -1)                           
        gray = cv2.cvtColor(hwc, cv2.COLOR_RGB2GRAY)               
        resized = cv2.resize(gray, (160, 100), interpolation=cv2.INTER_CUBIC)
        clipped = np.clip(resized, 0, 255).astype(np.uint8)         
        return clipped.reshape(100, 160, 1)

In [4]:
class PerceptionLayer:
    URGENCY_IMMINENT   =  150
    URGENCY_APPROACHING = 400
    SIDE_THRESHOLD = 15
    URGENCY_ENC = {"DISTANT": 0, "APPROACHING": 1, "IMMINENT": 2}
    SIDE_ENC    = {"CENTER": 0, "LEFT": 1, "RIGHT": 2}
    DIR_ENC     = {"PARALLEL TO": 0, "AWAY FROM": 1, "TOWARD": 2}
    HIT_ENC     = {None: 0, "HIT": 1, "FATAL": 2}
    MAX_PROJ    = 3

    ACTION_NAMES = {0: "LEFT", 1: "RIGHT"}

    def __init__(self):
        self.prev_health  = 100
        self.last_action  = None
        self.step_count   = 0

    def reset(self):
        self.prev_health = 100
        self.last_action = None
        self.step_count  = 0

    def update(self, state):
        self.step_count += 1

        player_y, health, hit = self._get_player_state(state)
        projectiles = self._get_projectiles(state, player_y)
        action = self.last_action
        state_text =  self._serialize(health, projectiles, hit, action)
        state_tag = self._get_state_tag(projectiles, hit, action)

        return state_text, state_tag
    
    def update_action(self, action):
        self.last_action = action
    
    def _get_player_state(self, state):
        vars = state.game_variables
        health = vars[0]
        player_y = vars[2]

        delta_h = health - self.prev_health
        self.prev_health = health
        hit = self._detect_hit(health, delta_h)

        return player_y, health, hit
    
    def _detect_hit(self, health, delta_h):
        if health <= 0:
            return "FATAL"
        if delta_h < 0:
            return "HIT"
        return None
    
    def _get_projectiles(self, state, player_y):
        projectiles = []
        for label in state.labels:
            if label.object_name == "DoomImpBall":
                delta_y = label.object_position_y - player_y
                delta_x = label.object_position_x
                projectiles.append({
                "delta_y": delta_y,
                "delta_x": delta_x,
                "side":    self._side(delta_y),
                "urgency": self._urgency(delta_x),
                })
        
        # Returns projectiles sorted by urgency (closest to player first)
        return sorted(projectiles, key=lambda p: abs(p["delta_x"]))
    
    def _side(self, delta_y: float):
        if abs(delta_y) < self.SIDE_THRESHOLD:
            return "CENTER"
        return "LEFT" if delta_y < 0 else "RIGHT"

    def _urgency(self, delta_x: float):
        dist = abs(delta_x)
        if dist < self.URGENCY_IMMINENT:
            return "IMMINENT"
        elif dist < self.URGENCY_APPROACHING:
            return "APPROACHING"
        return "DISTANT"
    
    def _evaluate_direction(self, action, threat_side):
        if threat_side == "CENTER":
            return "PARALLEL TO"
        if (threat_side == "RIGHT" and action == 1) or (threat_side == "LEFT" and action == 0):
            return "TOWARD"
        else:
            return "AWAY FROM"
    
    def _serialize(self, health, projectiles, hit, action):
        lines = []

        if not projectiles:
            return "No projectiles visible."
        else:
            if hit == "FATAL":
                lines.append("Health: 0/100 (DIED this step)")
            elif hit == "HIT":
                lines.append(f"Health: {int(health)}/100 (first hit, critical condition)")
            else:
                lines.append(f"Health: {int(health)}/100")
                
            lines.append(f"{len(projectiles)} projectile(s) detected:")
            for i, p in enumerate(projectiles, 1):
                if action is not None:
                    direction = self._evaluate_direction(action, p['side'])
                    lines.append(f"  {i}. {p['urgency'].upper()} threat on the {p['side']} side. The player MOVED {self.ACTION_NAMES.get(action)}. The player is MOVING {direction} this threat")
                else:
                    lines.append(f"  {i}. {p['urgency'].upper()} threat on the {p['side']} side")

        return "\n".join(lines)
    
    def _get_state_tag(self, projectiles, hit, action):
        hit_val = hit if hit else "NONE"
        act_val = self.ACTION_NAMES.get(action, "NONE")
        
        parts = [f"HIT:{hit_val}", f"ACT:{act_val}", f"NP:{len(projectiles)}"]
        
        for i, p in enumerate(projectiles):
            dir_val = self._evaluate_direction(action, p['side']) if action is not None else "NONE"
            p_str = f"P{i+1}:{p['urgency'][:4]},{p['side'][:4]},{dir_val[:4]}"
            parts.append(p_str)
            
        return " | ".join(parts)


In [5]:
class LLMPolicyAgent:
    CACHE_PATH = CACHE_PATH

    def __init__(self, model_name, backend_url):
        self.backend_url = backend_url
        self.model_name = model_name
        self.parse_failures = 0
        self._cache = {}
        self.payload = {
            "model" : self.model_name,
            "system" : self._build_system_prompt(),
            "prompt" : "",
            "stream" : False,
        }

        self._cache = self._load_cache()

    def decide(self, state_text, state_tag):
        key = hashlib.md5(state_tag.encode()).hexdigest()

        if key in self._cache:
            return self._cache[key]
        
        prompt = self._build_prompt(state_text)
        response = self._call_llm(prompt)
        action  = self._parse_action(response)

        self._cache[key] = action
        
        return action
    
    def _build_system_prompt(self):
        return """You are a policy model for a dodge game. The player must survive as long as possible by dodging fireballs shot by stationary enemies.
        The player can only move left or right.

        ### ALLOWED ACTIONS:
        - MOVE_LEFT: the player moves left.
        - MOVE_RIGHT: the player moves right.

        ### DECISION CRITERIA:
        - If a projectile is IMMINENT on your RIGHT side, move LEFT immediately.
        - If a projectile is IMMINENT on your LEFT side, move RIGHT immediately.
        - If a projectile is IMMINENT at CENTER, move in the direction with fewer total threats.
        - If multiple projectiles, prioritize the closest (listed first).
        - If no projectiles are visible, maintain current direction.

        ### OUTPUT FORMAT:
        Write the final line exactly as: ACTION: <action>"""
    
    def _build_prompt(self, state_text):
        return f"""### NOW EVALUATE: {state_text}"""

    def _call_llm(self, prompt):
        self.payload["prompt"] = prompt
        try:
            response = requests.post(self.backend_url, json= self.payload)

            # Check response status
            if response.status_code == 200:
                return response.json()["response"]
            else:
                print(f"Ollama returned error: {response.status_code}")
                return ""
        except requests.exceptions.ConnectionError:
            print("Failed to connect to Ollama backend.")
            return ""     
    
    def _parse_action(self, response):
        clean = response.strip().upper()
        if "MOVE_LEFT" in clean:
            return 0
        elif "MOVE_RIGHT" in clean:
            return 1
        else:
            # fallback: random action if parsing fails + log the failure
            self.parse_failures += 1
            return random.randint(0, 1)
        
    def _load_cache(self):
        if os.path.exists(self.CACHE_PATH):
            with open(self.CACHE_PATH, "r") as f:
                return json.load(f)
        return {}

    def _save_cache(self):
        with open(self.CACHE_PATH, "w") as f:
            json.dump(self._cache, f)

In [6]:
def make_eval_env():
    def _init():
        env = VizDoomGym(render=False)
        env = Monitor(env)
        return env
    return _init

In [7]:
def evaluate_agent(agent_fn, n_episodes=100, needs_vecenv=True):
    if needs_vecenv:
        env = DummyVecEnv([make_eval_env()])
        env = VecFrameStack(env, n_stack=4, channels_order='last')
        
        survival_times = []
        episode_rewards = []

        for _ in tqdm(range(n_episodes)):
            obs = env.reset()
            terminated    = False
            episode_reward = 0.0

            while not terminated:
                action = agent_fn(obs, env)
                obs, reward, dones, infos = env.step(action)
                episode_reward += reward[0]         # VecEnv restituisce array
                terminated = dones[0]
                if terminated and "survival_time_tics" in infos[0]:
                    survival_times.append(infos[0]["survival_time_tics"])

            episode_rewards.append(episode_reward)

    else:
        env = VizDoomGym(render=False)
        survival_times  = []
        episode_rewards = []

        for _ in tqdm(range(n_episodes)):
            obs, _ = env.reset()
            terminated     = False
            steps          = 0
            episode_reward = 0.0

            while not terminated:
                action = agent_fn(obs, env)
                obs, reward, terminated, _, _ = env.step(action)
                episode_reward += reward
                steps += 1

            survival_times.append(steps)
            episode_rewards.append(episode_reward)

    env.close()
    return (
        np.mean(survival_times), np.std(survival_times),
        np.mean(episode_rewards), np.std(episode_rewards),
    )

# Variant 1 and 3: PPO baseline and PPO with LLM reward
def ppo_agent_fn(obs, env, ppo_model):
    action, _ = ppo_model.predict(obs)
    return action

# Variant 2: LLM action
def llm_agent_fn(obs, env, perception, agent):
    state = env.game.get_state()

    if state is None:
        return random.randint(0, 1)
    
    state_text, state_tag = perception.update(state)
    action = agent.decide(state_text, state_tag)
    perception.update_action(action)   
    
    return action

In [8]:
# Comparison Loop
BASELINE_PATH = 'models/take_cover_best/best_model.zip'
REWARD_PATH = 'models/take_cover_best_llm_reward/best_model.zip'

try:
    ppo_baseline_model = PPO.load(BASELINE_PATH)
    reward_agent = PPO.load(REWARD_PATH)
except Exception as e:
    print(f"ERROR during model loading : {e}")

perception_layer = PerceptionLayer()
policy_agent = LLMPolicyAgent(model_name="llama3.2:1b", backend_url="http://localhost:11434/api/generate")

agents = [
    ("PPO baseline",    lambda obs, env, m=ppo_baseline_model: ppo_agent_fn(obs, env, m),True),
    ("PPO+LLM reward",  lambda obs, env, m=reward_agent: ppo_agent_fn(obs, env, m), True),
    ("LLM policy",      lambda obs, env, p=perception_layer, a=policy_agent: llm_agent_fn(obs, env, p, a), False),

]

print(f"\n{'Agent':<20} {'Survival (tics)':>16} {'Reward':>16}")
print("-" * 56)

for name, fn, needs_vec in agents:
    surv_mean, surv_std, rew_mean, rew_std = evaluate_agent(
        fn, n_episodes=100, needs_vecenv=needs_vec
    )
    print(f"{name:<20} "
          f"{surv_mean:>7.1f} ± {surv_std:<6.1f} "
          f"{rew_mean:>7.1f} ± {rew_std:<6.1f}")


Agent                 Survival (tics)           Reward
--------------------------------------------------------


100%|██████████| 100/100 [01:12<00:00,  1.37it/s]


PPO baseline           526.5 ± 330.0    518.7 ± 329.5 


100%|██████████| 100/100 [01:25<00:00,  1.17it/s]


PPO+LLM reward         582.6 ± 323.2    574.9 ± 322.9 


100%|██████████| 100/100 [00:52<00:00,  1.92it/s]

LLM policy              82.1 ± 17.8     320.3 ± 70.8  
